    Storing Data in Vector Databases

    Steps:
1.     Generate Embeddings: Use pre-trained models to create vector representations of your data (text, images, or audio).
2.     Organize Metadata: Store metadata like file names, descriptions, or labels alongside the embeddings to help identify the original data during retrieval.
3.     Initialize the Vector Database: Set up a vector database instance to manage and query your embeddings.
4.     Insert Data: Store embeddings and their corresponding metadata in the database.
5.     Query: Perform similarity searches using a query embedding.


    Đây là một ví dụ hoàn chỉnh và chuẩn xác về một quy trình RAG mini: từ việc tạo Embedding, lưu trữ vào Vector Database (ChromaDB) cho đến truy vấn tìm kiếm sự tương đồng.
    Dưới đây là giải thích chi tiết những gì mã này thực hiện và một vài lưu ý quan trọng để bạn vận hành nó tốt hơn:
    1. Phân tích các bước chính
        Lưu trữ bền vững (persist_directory): Bằng cách chỉ định thư mục ./chroma_text_db, dữ liệu của bạn sẽ không bị mất đi khi tắt chương trình. Chroma sẽ tạo một thư mục tại đó để lưu các file index và vector.
        Đối tượng Document: Bạn đã sử dụng đúng cấu trúc langchain_core. Việc đưa thêm metadata cực kỳ quan trọng vì sau này bạn có thể lọc dữ liệu (filtering) dựa trên nguồn (source), ngày tháng, v.v.
        add_documents: Phương thức này sẽ:
            Dùng embedding_model biến page_content thành các vector.
            Đẩy cả vector và metadata vào database.
        similarity_search với k=2:
            Biến câu hỏi "What is Tesla's mission?" thành vector.
            So sánh vector này với kho dữ liệu.
            Lấy ra 2 kết quả (k=2) có khoảng cách gần nhất (giống nhất).
    2. Các điểm cộng của đoạn code này
        Đúng chuẩn mới: Sử dụng các gói tách biệt (_community, _huggingface, _core).
        Dễ mở rộng: Bạn có thể dễ dàng thay danh sách documents bằng dữ liệu đọc từ PDF hay Web.
        Tốc độ: Mô hình all-MiniLM-L6-v2 rất nhẹ, phù hợp chạy local.
    3. Những lưu ý quan trọng khi chạy thực tế
        A. Tránh trùng lặp dữ liệu (Duplication)
        Nếu bạn chạy script này 10 lần, bảng của bạn sẽ có 30 dòng (vì mỗi lần chạy nó lại add_documents thêm 3 cái y hệt).
            Giải pháp: Kiểm tra xem database đã có dữ liệu chưa trước khi thêm, hoặc xóa thư mục ./chroma_text_db trước khi chạy lại nếu muốn làm mới.
        B. Cài đặt thư viện
        Đảm bảo bạn đã cài các gói sau để code chạy mượt:
            pip install langchain-community langchain-huggingface langchain-core chromadb sentence-transformers
        C. Lấy thêm điểm tin cậy (Score)
        Nếu bạn muốn biết kết quả tìm được "giống" bao nhiêu %, bạn có thể dùng similarity_search_with_score thay vì similarity_search:
            results = chroma_db.similarity_search_with_score(query=query_text, k=2)
            for doc, score in results:
                print(f"Score: {score} - Content: {doc.page_content}")
        Lưu ý: Trong Chroma, score càng thấp nghĩa là khoảng cách càng ngắn -> càng giống.
    4. Bước tiếp theo (The "G" in RAG)
    Đoạn code của bạn đã làm tốt phần R (Retrieval - Truy xuất). Để hoàn thiện RAG, bạn chỉ cần thêm bước G (Generation):
        Lấy nội dung từ results.
        Đưa vào một Prompt: "Dựa vào các thông tin sau: {nội dung kết quả}, hãy trả lời câu hỏi: {câu hỏi}".
        Gửi Prompt đó cho một LLM (như GPT-4 hoặc Llama-3) để nhận câu trả lời cuối cùng.

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Step 1: Load Embedding Model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Step 2: Initialize ChromaDB Vector Store
chroma_db = Chroma(
    persist_directory="./chroma_text_db",  # Directory to store Chroma data
    embedding_function=embedding_model
)

# Step 3: Example Text Data
documents = [
    Document(
        page_content="Elon Musk leads Tesla and SpaceX.",
        metadata={"source": "1"}
    ),
    Document(
        page_content="Tesla's mission is to accelerate the world's transition to sustainable energy.",
        metadata={"source": "2"}
    ),
    Document(
        page_content="SpaceX aims to make space travel accessible to humanity.",
        metadata={"source": "3"}
    ),
]

# Step 4: Add Documents to ChromaDB
chroma_db.add_documents(documents)

# Step 5: Perform a Similarity Query
query_text = "What is Tesla's mission?"
results = chroma_db.similarity_search(query=query_text, k=2)

print(results)

# Step 6: Display Results
for idx, result in enumerate(results):
    print(f"Result {idx}:")
    print(f"Content {result.page_content}")
    print(f"Metadata {result.metadata}")

[Document(id='14ea5b9c-9824-4d20-8b28-af906f7087b2', metadata={'source': '2'}, page_content="Tesla's mission is to accelerate the world's transition to sustainable energy."), Document(id='47005497-f1e4-4524-8c9a-8fc18e5196d8', metadata={'source': '1'}, page_content='Elon Musk leads Tesla and SpaceX.')]
Result 0:
Content Tesla's mission is to accelerate the world's transition to sustainable energy.
Metadata {'source': '2'}
Result 1:
Content Elon Musk leads Tesla and SpaceX.
Metadata {'source': '1'}


    Vector Search vs Semantic Search

    Vector Search (nó phục vụ cho Semantic Search, chứ nếu nó trả ra kết quả luôn thì kết quả không tốt)
        Vector Search converts the query to embeddings using the same vectorization model as the database.
        Vector Search uses SentenceTransformer for embedding generation.

    Semantic Search
        Semantic Search uses an LLM to understand and reformulate queries.
        Semantic Search uses a GPT-like model for deeper semantic understanding.

 Semantic Search


    Đoạn code bạn cung cấp là một ví dụ điển hình về hệ thống RAG (Retrieval-Augmented Generation) cơ bản. Mục tiêu của nó là tìm kiếm phim dựa trên ý nghĩa (ngữ nghĩa) của câu hỏi thay vì chỉ khớp từ khóa đơn thuần.
    Dưới đây là giải thích chi tiết từng bước:
    1. Khởi tạo dữ liệu (Step 1)
    Bạn tạo một danh sách các bộ phim gồm tiêu đề và mô tả. Đây là "kiến thức" mà AI sẽ sử dụng để trả lời.
    2. Tạo Embeddings và Cơ sở dữ liệu Vector (Step 2)
        Embeddings (all-MiniLM-L6-v2): Đây là quá trình biến các câu văn bản (mô tả phim) thành các dãy số (vector). Các câu có nội dung tương tự nhau sẽ có các dãy số nằm gần nhau trong không gian toán học.
        Documents: Chuyển đổi dữ liệu thô thành định dạng Document của LangChain để bộ lưu trữ có thể đọc được.
        ChromaDB: Đây là một cơ sở dữ liệu vector (Vector Database). Nó lưu trữ các vector mô tả phim để sau này chúng ta có thể so sánh với vector của câu hỏi từ người dùng.
    3. Thiết lập mô hình ngôn ngữ lớn - LLM (Step 3)
        Sử dụng mô hình google/flan-t5-large. Đây là một mô hình ngôn ngữ từ Google có khả năng hiểu và trả lời văn bản.
        HuggingFacePipeline: Giúp tích hợp mô hình từ thư viện Hugging Face vào khung làm việc của LangChain.
    4. Tạo Retriever và QA Chain (Step 4)
        Retriever: Nhiệm vụ của nó là khi nhận được một câu hỏi, nó sẽ chạy vào ChromaDB để tìm ra 2 bộ phim (k=2) có mô tả giống với ý nghĩa câu hỏi nhất.
        RetrievalQA: Đây là "sợi dây" kết nối tất cả lại. Quy trình vận hành như sau:
            Nhận câu hỏi.
            Tìm tài liệu liên quan thông qua Retriever.
            Đưa tài liệu đó và câu hỏi vào LLM để LLM tổng hợp câu trả lời.
        Custom Prompt: Bạn tạo một hàm để ép LLM trả lời theo định dạng mong muốn (liệt kê danh sách phim từ tập dữ liệu).
    5. Thực hiện truy vấn (Step 5)
        Câu hỏi: "Find me movies about astronauts struggling to survive in space." (Tìm phim về phi hành gia đấu tranh sinh tồn trong không gian).
        Cách hoạt động:
            Hệ thống sẽ không chỉ tìm từ "space", nó hiểu "struggling to survive" và "astronauts" liên quan đến các mô tả của phim The Martian hay Gravity.
            Nó lấy thông tin từ các phim đó và đưa cho LLM.
            LLM sẽ đọc và trả về kết quả cuối cùng.

    Một số lưu ý nhỏ để code chạy tốt hơn:
        Sử dụng Prompt Template: Trong code của bạn, custom_prompt đang được dùng để biến đổi câu hỏi trước khi đưa vào qa_chain. Tuy nhiên, cách chuẩn của LangChain là truyền một prompt_template trực tiếp vào RetrievalQA.from_chain_type.
        Thiết bị: device=-1 nghĩa là đang chạy bằng CPU. Nếu bạn có GPU (Card đồ họa), hãy đổi thành device=0 để chạy nhanh hơn.
        Thư viện: Bạn đang dùng langchain_classic, hãy lưu ý rằng các phiên bản mới của LangChain thường khuyến khích dùng create_retrieval_chain thay vì RetrievalQA (vốn đã cũ).

In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_classic.chains import RetrievalQA


# Step 1: Set up the movies dataset
movies = [
    {
        "title": "Interstellar",
        "description": "A group of astronauts travels through a wormhole in search of a new home for humanity."
    },
    {
        "title": "The Martian",
        "description": "An astronaut becomes stranded on Mars and struggles to survive while awaiting rescue."
    },
    {
        "title": "Inception",
        "description": "A thief uses dream-sharing technology to steal secrets but takes on an impossible heist."
    },
    {
        "title": "Gravity",
        "description": "Two astronauts work together to survive after an accident leaves them stranded in space."
    },
    {
        "title": "Apollo 13",
        "description": "NASA must devise a strategy to return Apollo 13 to Earth safely after an explosion on the spacecraft."
    }
]

# Step 2: Generate embeddings using Hugging Face Embeddings
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Convert movie data to Documents
documents = [
    Document(
        page_content=movie["description"],
        metadata={"title": movie["title"]}
    )
    for movie in movies
]

# Initialize ChromaDB with movie documents
vector_store = Chroma.from_documents(documents, embedding_model)

# Step 3: Set up a Hugging Face pipeline for LLM
llm_model_name = "google/flan-t5-large"  # Replace with "zypher" if available on HF Hub
hf_pipeline = pipeline(
    "text2text-generation",
    model=llm_model_name,
    device=0,
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Step 4: Create Retriever
# k=2: Chỉ lấy 2 phim giống nhất để đưa vào ngữ cảnh (tránh làm đầy bộ nhớ LLM)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Custom prompt to ensure specific results
def custom_prompt(query: str):
    return f"""
        You are a helpful assistant tasked with retrieving movie titles based on descriptions.
        Query: {query}
        From the following dataset, only provide the movie titles that match:
        Dataset:
        {', '.join([doc.metadata['title'] for doc in documents])}
        Response:
    """

# Step 4: Define the QA chain with the custom prompt
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)

# Step 5: Perform semantic search
semantic_query = "Find me movies about astronauts struggling to survive in space."
custom_query = custom_prompt(semantic_query)
semantic_results = qa_chain.run(custom_query)

print("Semantic Search Results:")
print(semantic_results)



Device set to use cpu


Semantic Search Results:
Interstellar, The Martian, Inception, Gravity, Apollo 13
